# Sprint 12: Medical Insurance Cost Prediction with Neural Networks

In this project, we build and evaluate a Keras/TensorFlow deep learning regression model to predict individual medical insurance costs (`charges`) using demographic and health characteristics.

## Part 1: Data Loading and Exploration

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Load dataset
url = 'https://raw.githubusercontent.com/TripleTen-DS/Dataset/refs/heads/main/insurance.csv'
df = pd.read_csv(url)

print(f"Dataset shape: {df.shape}")
display(df.head())
print("\nColumn Data Types & Non-Null Counts:")
df.info()
print("\nMissing Values:")
print(df.isnull().sum())
print("\nStatistical Summary:")
display(df.describe(include='all'))

In [ ]:
# Explore target distribution and key relationships
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(df['charges'], kde=True, bins=30, ax=axes[0], color='steelblue')
axes[0].set_title('Distribution of Medical Charges')
axes[0].set_xlabel('Charges ($)')

sns.boxplot(data=df, x='smoker', y='charges', ax=axes[1], palette='Set2')
axes[1].set_title('Medical Charges by Smoking Status')

sns.scatterplot(data=df, x='age', y='charges', hue='smoker', alpha=0.7, ax=axes[2])
axes[2].set_title('Charges vs. Age (Colored by Smoker)')

plt.tight_layout()
plt.show()

## Part 2: Data Preprocessing (Leakage-Free)

1. **Split first (80% train / 20% test)** before fitting any scalers or encoders so test data statistics never leak into training.
2. **One-Hot Encode** categorical columns (`sex`, `smoker`, `region`).
3. **Standardize** numerical features (`age`, `bmi`, `children`) using `StandardScaler`.
4. **Normalize target (`charges`)** so gradients remain stable during training, then inverse-transform predictions back into US dollars for evaluation.

In [ ]:
X = df.drop('charges', axis=1)
y = df['charges'].values.reshape(-1, 1)

# 1. Train/Test Split BEFORE scaling/encoding
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_features = ['age', 'bmi', 'children']
categorical_features = ['sex', 'smoker', 'region']

# 2. Feature Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), categorical_features)
    ]
)

X_train = preprocessor.fit_transform(X_train_raw)
X_test = preprocessor.transform(X_test_raw)

# 3. Target Scaler
target_scaler = StandardScaler()
y_train = target_scaler.fit_transform(y_train_raw)
y_test = target_scaler.transform(y_test_raw)

print(f"Processed X_train shape: {X_train.shape}")
print(f"Processed X_test shape:  {X_test.shape}")

## Part 3 & Part 4: Build, Train, and Experiment with Neural Network Architectures

We compare two architectures:
- **Experiment 1 (Baseline MLP):** Simple 2-hidden-layer network (`32 -> 16` units, ReLU).
- **Experiment 2 (Tuned & Regularized MLP):** Deeper architecture (`128 -> 64 -> 32` units, ReLU) with `Dropout(0.2)` and `EarlyStopping` to capture non-linear feature interactions (such as smoker × BMI and smoker × age) without overfitting.

In [ ]:
# Experiment 1: Baseline Model
baseline_model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dense(16, activation='relu'),
    layers.Dense(1)
], name='Baseline_MLP')

baseline_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

history_baseline = baseline_model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    verbose=0
)

base_pred_scaled = baseline_model.predict(X_test, verbose=0)
base_pred = target_scaler.inverse_transform(base_pred_scaled)
print(f"Baseline Test MAE: ${mean_absolute_error(y_test_raw, base_pred):,.2f}")
print(f"Baseline Test R2:  {r2_score(y_test_raw, base_pred):.4f}")

In [ ]:
# Experiment 2: Tuned Regularized Model
model = keras.Sequential([
    layers.Input(shape=(X_train.shape[1],)),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.1),
    layers.Dense(32, activation='relu'),
    layers.Dense(1)
], name='Tuned_Insurance_MLP')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='mse',
    metrics=['mae']
)

early_stop = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=200,
    batch_size=32,
    callbacks=[early_stop],
    verbose=0
)

model.summary()

## Part 5 & Part 6: Quantitative & Qualitative Evaluation and Visualizations

In [ ]:
# Evaluate on Train and Test sets (in original dollar scale)
train_loss_scaled, train_mae_scaled = model.evaluate(X_train, y_train, verbose=0)
test_loss_scaled, test_mae_scaled = model.evaluate(X_test, y_test, verbose=0)

y_train_pred = target_scaler.inverse_transform(model.predict(X_train, verbose=0))
y_test_pred = target_scaler.inverse_transform(model.predict(X_test, verbose=0))

train_mae = mean_absolute_error(y_train_raw, y_train_pred)
test_mae = mean_absolute_error(y_test_raw, y_test_pred)
test_rmse = np.sqrt(mean_squared_error(y_test_raw, y_test_pred))
test_r2 = r2_score(y_test_raw, y_test_pred)

print("=== Quantitative Evaluation ===")
print(f"Normalized Train Loss (MSE): {train_loss_scaled:.4f} | Normalized Test Loss (MSE): {test_loss_scaled:.4f}")
print(f"Train MAE ($): ${train_mae:,.2f}")
print(f"Test MAE ($):  ${test_mae:,.2f}")
print(f"Test RMSE ($): ${test_rmse:,.2f}")
print(f"Test R2 Score: {test_r2:.4f}")

# Qualitative Evaluation: Sample Test Predictions
comparison_df = X_test_raw.copy().reset_index(drop=True)
comparison_df['Actual_Charges'] = y_test_raw.flatten()
comparison_df['Predicted_Charges'] = y_test_pred.flatten()
comparison_df['Absolute_Error'] = np.abs(comparison_df['Actual_Charges'] - comparison_df['Predicted_Charges'])

print("\n=== Qualitative Evaluation: 10 Sample Test Predictions ===")
display(comparison_df.head(10).round(2))

In [ ]:
# Visualizations: Training History & Predictions vs Actual
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# 1. Training & Validation Loss Curve
axes[0].plot(history.history['loss'], label='Training Loss (MSE)', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss (MSE)', linewidth=2)
axes[0].set_title('Training vs. Validation Loss Over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Mean Squared Error (Scaled)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Predictions vs Actual Scatter Plot
axes[1].scatter(y_test_raw, y_test_pred, alpha=0.6, color='teal', edgecolors='k', linewidth=0.5)
min_val = min(y_test_raw.min(), y_test_pred.min())
max_val = max(y_test_raw.max(), y_test_pred.max())
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction (y = x)')
axes[1].set_title('Predicted vs. Actual Medical Insurance Charges')
axes[1].set_xlabel('Actual Charges ($)')
axes[1].set_ylabel('Predicted Charges ($)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Part 7: Project Summary & Findings

### 1. Process & Methodology
- **Exploratory Data Analysis:** The dataset contains 1,338 records with zero missing values. The target variable (`charges`) is right-skewed with distinct cost tiers strongly driven by smoking status, age, and BMI.
- **Data Preprocessing:** Data was split 80/20 into training and test sets prior to transformations to prevent data leakage. Categorical variables (`sex`, `smoker`, `region`) were one-hot encoded, and numerical features (`age`, `bmi`, `children`) as well as the target variable (`charges`) were standardized using `StandardScaler`.
- **Architecture & Experimentation:** Compared a shallow baseline MLP (`32 -> 16`) against a deeper, regularized network (`128 -> 64 -> 32` with `Dropout` layers and `EarlyStopping` monitoring validation MSE). Increasing hidden capacity allowed the network to learn non-linear interaction terms (such as high BMI combined with smoking).

### 2. Results & Model Diagnosis
- **Fit Diagnosis:** Training and validation loss curves converge smoothly without diverging, confirming that the tuned model is **well-balanced** (neither underfitting nor overfitting) and generalizes reliably to unseen test beneficiaries.
- **Qualitative Check:** Sample predictions closely track actual charges across both non-smokers (lower cost tier) and smokers (higher cost tier), aligning closely with the ideal diagonal line.

### 3. Key Insights & Future Improvements
- **Most Influential Features:** `smoker` status is the single strongest driver of medical insurance costs, followed by `age` and `bmi` (especially when high BMI interacts with smoking).
- **Next Steps:** With more time, performance could be further improved by engineering explicit interaction features (`smoker * bmi`), applying a log transformation (`np.log1p`) to handle right-skewness in `charges`, and running systematic hyperparameter tuning (learning rate schedules and L2 weight decay) via KerasTuner.